# Aging chains and coflows in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/09-chains.ipynb)

This notebook is the Python companion to the [Aging chains and coflows](https://waelrash1.github.io/loopbricks/#/learn/chains) module. You will follow a population through the stages of life, see why a change in births takes decades to work through, watch fast hiring dilute a workforce, and track an attribute (experience) that travels with the people who carry it. It uses four LoopBricks molecules: Aging Chain, Aging Chain with PDY, Coflow and Coflow with Experience.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. A chain of stages

An **aging chain** splits one stock into stages that items pass through in order: here youth, working age and old age. Each stage is a first-order delay, so the flow out of a stage is the stage's contents divided by its average duration. Unlike a plain delay, a chain lets you see, and act on, each stage separately.

In **Aging Chain**, 20 people are born each year, stay young for 18 years on average, work for 45 and live 15 years after that.

In [ ]:
ag = lb.molecule("aging")
print(ag.equations())
pop = ag.run()
ag.plot(pop)

print("\nAt the start: ", pop[["Youth", "Working age", "Elderly"]].iloc[0].round(0).to_dict())
print("After 60 years:", pop[["Youth", "Working age", "Elderly"]].iloc[-1].round(0).to_dict())
print("Births × duration:", {"Youth": 20 * 18, "Working age": 20 * 45, "Elderly": 20 * 15})

In balance each stage holds its inflow times the time spent in it, which is Little's law from the delays module applied stage by stage. The model starts exactly there, with 360, 900 and 300 people, so the run is flat: every year each stage gains as many people as it loses (20 a year all the way through). Starting a model in equilibrium is good practice, because then any movement you see is caused by the change you make, not by the model settling down.

Now cut births from 20 to 8 a year (the *Birth decline* preset).

In [ ]:
dec = ag.run(preset="Birth decline")
ag.plot(dec)
for stage in ("Youth", "Working age", "Elderly"):
    print(f"{stage:<12}: {dec[stage].iloc[0]:4.0f} at the start, {dec.loc[10.0, stage]:4.0f} in year 10, "
          f"{dec[stage].iloc[-1]:4.0f} in year 60, first 10% down in year {dec.index[dec[stage] < 0.9 * dec[stage].iloc[0]][0]}")

The youth group shrinks at once and is 10% down after 3.3 years. The working-age group hardly notices for a decade (874 in year 10) and is 10% down only in year 21. The elderly group is still at 298 in year 10 and takes until year 34 to fall by 10%. The people retiring over those decades were born long before the change: the chain carries the past forward, and each stage passes the shock on later and more gently than the one before. This is why a country's age structure keeps shifting for generations after its birth rate changes.

**Try it.** With births at 8 a year, how does the number of elderly per person under retirement age change between year 0 and year 60?

<details><summary>Show a worked answer</summary>

It rises from 0.24 (300 elderly for 1,260 younger people) to 0.28 in year 20 and 0.30 by year 40, where it levels off. Fewer births mean fewer future workers, but the elderly were already in the chain, so each working-age person ends up supporting more retirees.
</details>

In [ ]:
ratio = dec["Elderly"] / (dec["Youth"] + dec["Working age"])
print(ratio.loc[[0.0, 20.0, 40.0, 60.0]].round(2))

## 2. Hiring fast dilutes a workforce

The same structure works for staff. In **Aging Chain with PDY** (production, from rookies, experienced staff and veterans), a firm hires 20 people a year. Rookies take 3 years to learn the job and produce half as much as an experienced person; veterans produce 0.8. The firm starts in balance. What happens if it doubles hiring to 40 a year to grow?

In [ ]:
pdy = lb.molecule("agingPDY")
fig, ax = plt.subplots(figsize=(8, 4))
for h in (20, 40):
    run = pdy.run(hiring=h)
    staff = run[["Rookies", "Experienced", "Veterans"]].sum(axis=1)
    (run["Production"] / staff).plot(ax=ax, label=f"hiring {h} a year")
    print(f"hiring {h}: staff {staff[0.0]:.0f} -> {staff.iloc[-1]:.0f}, output per person "
          f"{(run['Production'] / staff)[0.0]:.3f} at the start, lowest {(run['Production'] / staff).min():.3f}")
ax.set_ylabel("production per person"); ax.legend(); ax.grid(alpha=0.3)

Doubling hiring grows the firm from 720 to 1365 people, but for the first few years the newcomers are mostly rookies. Output per person drops from 0.914 to 0.894 before slowly recovering as the rookies mature. The dip is small here because rookies are only 3 years in the chain; in a firm where people need ten years to become productive, or where experienced staff must also spend time training rookies, growth can lower total output for a while.

**Try it.** In the fast-hiring run, when is the share of rookies highest, and how high does it get?

<details><summary>Show a worked answer</summary>

The rookie share rises from 8.3% to a peak of 13.4% in year 6.1, then falls back (8.8% in year 60) as the larger intake works its way into the experienced stage. In the new balance it returns to 8.3%, because the shares depend only on how long people spend in each stage (3, 25 and 8 years), not on how many are hired. Output per person is lowest at almost the same moment (year 5.2).
</details>

In [ ]:
run = pdy.run(hiring=40)
rookie_share = run["Rookies"] / run[["Rookies", "Experienced", "Veterans"]].sum(axis=1)
print(f"Rookie share: {rookie_share[0.0]:.1%} at the start, peak {rookie_share.max():.1%} in year {rookie_share.idxmax()}, "
      f"{rookie_share.iloc[-1]:.1%} in year 60")

## 3. Carrying an attribute with a coflow

Sometimes you care less about which stage people are in than about a quality they carry, such as skill, age or cost. A **coflow** tracks it with a second stock that mirrors the first. In **Coflow**, the main stock is headcount; the coflow is the team's total years of experience. Experience enters with hires (`Hire experience` each), grows by one year per person per year, and leaves with each leaver at the current average. The average is total experience divided by headcount. (The molecule also lists `Avg experience`, which is simply that average again, kept for the chart; you can ignore it.)

In [ ]:
cf = lb.molecule("coflow")
print(cf.equations(), "\n")
fig, ax = plt.subplots(figsize=(8, 4))
for label in ("Steady team", "Rapid hiring (dilution)", "Hire veterans"):
    run = cf.run(preset=label)
    run["Experience per head"].plot(ax=ax, label=f"{label} {cf.preset(label)}")
    print(f"{label:<24}: experience per head {run.loc[5.0, 'Experience per head']:.2f} in year 5, {run['Experience per head'].iloc[-1]:.2f} in year 60")
ax.set_ylabel("years of experience per head"); ax.legend(); ax.grid(alpha=0.3)

The steady team starts in balance (48 people with 288 years of experience between them) and stays at 6 years per head throughout. That is exactly the average tenure: in a well-mixed stock, the average time people have spent inside equals the average time they stay. Hiring veterans with 5 years of experience adds those 5 years on top, giving 11. Rapid hiring (18 a year instead of 8) pulls the average down to 4.4 years in year 5, while the team more than doubles, but it ends at the same 6: in this model, fast growth dilutes experience only while the growth lasts.

**Try it.** What average experience do you expect if people stay 12 years instead of 6 (`avg_tenure=12`)? Is the run long enough to show it?

<details><summary>Show a worked answer</summary>

Expect 12 years, the new tenure. After 60 years the run shows 11.78, still climbing: with a 12-year tenure both the headcount and the experience adjust on a 12-year time scale, and 60 years is only five of those. Run it for 120 years and it reaches 12.00.
</details>

In [ ]:
for T in (60, 120):
    print(f"T = {T}: experience per head {cf.run(T=T, avg_tenure=12)['Experience per head'].iloc[-1]:.2f}")

## 4. Tracking the average directly

**Coflow with Experience** writes the same idea differently: instead of a stock of total experience, it keeps a stock of *average* experience. The average rises by one year each year, and new hires pull it towards their own experience in proportion to how many of them there are compared with the workforce. Both formulations give the same answer; this one is handy when only the average matters.

The firm starts in balance with 80 people, 10 hires a year and an average of 8 years. Then hiring doubles to 20 a year.

In [ ]:
ce = lb.molecule("coflowExperience")
fig, ax = plt.subplots(figsize=(8, 4))
for label in ("Steady team", "Hiring surge (dilution)", "Hire veterans"):
    run = ce.run(preset=label)
    run["Average experience"].plot(ax=ax, label=f"{label} {ce.preset(label)}")
ax.set_ylabel("average experience (years)"); ax.legend(); ax.grid(alpha=0.3)

surge = ce.run(preset="Hiring surge (dilution)")
print(surge.loc[[0.0, 5.0, 10.0, 20.0, 60.0], ["Workforce", "Average experience"]].round(2))

The surge doubles the workforce to 160, and the flood of beginners drags the average experience down from 8 to just over 6 within about 6 years. It then climbs back to 8 (7.98 by year 60): the same answer as section 3, average experience equal to average tenure (8 years here), however big the firm.

**Try it.** With the same surge, hire people who already have 5 years of experience (the *Hire veterans* preset). Does the dilution still happen, and where does the average end up?

<details><summary>Show a worked answer</summary>

There is no dip: the average rises from the start, to 9.35 in year 5 and 12.98 by year 60, on its way to 13, which is tenure plus the experience each hire brings (8 + 5). Hiring people with experience can protect a growing team from dilution, at a price.
</details>

In [ ]:
vets = ce.run(preset="Hire veterans")
print(vets.loc[[0.0, 5.0, 20.0, 60.0], "Average experience"].round(2))

## Where next

- Back in the app: the [Aging chains and coflows](https://waelrash1.github.io/loopbricks/#/learn/chains) module has the diagrams and more exercises, including promotion chains and cost coflows.
- Each stage in a chain is a delay: see the [Delays and perception](https://waelrash1.github.io/loopbricks/#/learn/delays) module and its notebook.
- Workforce chains drive the labour cycles in [Cycles in business, labour and commodity markets](https://waelrash1.github.io/loopbricks/#/learn/cycles). For more on aging chains and coflows, see Sterman 2000, ch. 12.